## Introduction



In this notebook, we will explore the integration of Large Language Models (LLMs) like Google's Gemini and vector databases like Pinecone to create a powerful Retrieval-Augmented Generation (RAG) system. By leveraging prompt engineering, we will calibrate the LLM to act as an equity research expert, capable of answering questions related to the content of a PDF document. This approach allows us to enhance the LLM's responses by providing it with relevant context retrieved from the vector database, ensuring more accurate and contextually appropriate answers.

## Importing Libraries

In [30]:
import textwrap
import numpy as np
import pandas as pd

from typing import List

import google.generativeai as genai
import google.ai.generativelanguage as glm

from PyPDF2 import PdfReader

# Importing the CharacterTextSplitter class from the langchain library to split the text into chunks
from langchain.text_splitter import CharacterTextSplitter

from pinecone import Pinecone


from IPython.display import Markdown

import getpass
import os


## Google Gemini (LLM) model Configuration

In [31]:
GOOGLE_API_KEY=getpass.getpass()
genai.configure(api_key=GOOGLE_API_KEY)

## Pinecone (Vector DB) Configuration

In [32]:
pc = Pinecone("492fe419-7850-4384-8dc4-d2019c9d1ab2")

### PDF Content Extraction and Text Chunking

This code extracts text from a PDF file and then splits it into manageable chunks for further processing. Here’s a breakdown of each step:

1. **PDF Content Extraction**:
   - The code initializes an empty string `pdf_content` to hold the extracted text.
   - It loops through a list of PDF files (`pdf_docs`), opening each with `PdfReader`.
   - For each page in the PDF, the text is extracted and appended to `pdf_content`.

2. **Text Chunking**:
   - The extracted content is split into smaller chunks to facilitate easier handling in later stages (e.g., for NLP tasks or processing with models).
   - A `CharacterTextSplitter` is used to split the text into segments of up to 2000 characters, with an overlap of 200 characters between each chunk to preserve context.
   
The output `chunks` variable holds the list of text segments, each of which is 2000 characters or fewer, overlapping by 200 characters.

In [ ]:
# Extract the content of the PDF
pdf_content = ""
# Loop through the PDF files
pdf_docs = ["equity research report.pdf"]
for pdf in pdf_docs:
    # Read the PDF file
    pdf_reader = PdfReader(pdf)
    # Loop through the pages of the PDF file
    for page in pdf_reader.pages:
        # Extract the text from the PDF page and add it to the pdf_content variable
        pdf_content += page.extract_text()
# st.write(pdf_content)
# Get chunks of the content
# Split the text into chunks of 2000 characters with an overlap of 200 characters
text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=2000,
    chunk_overlap=200,
    length_function=len,
)
# Split the text into chunks of 2000 characters with an overlap of 200 characters
chunks = text_splitter.split_text(pdf_content)

In [ ]:
df = pd.DataFrame(chunks)
df.columns = ["Content"]

,Content
0,Amr Ghribi\namrou.ghribi@esprit.tn | (+49) 1...
1,precision.\nDeployed on Vercel with minimal la...


### Embedding Generation and Vector Indexing

This code generates embeddings for text passages and indexes them for efficient querying, typically used in natural language processing applications.

1. **Embedding Generation**:
   - Using the `pc.inference.embed` function with the `"multilingual-e5-large"` model, the code generates embeddings for each text passage in the `df['Content']` DataFrame column.
   - Each passage is treated as an "input_type" of `"passage"` to customize embeddings based on context.

2. **Vector Creation**:
   - For each embedding, a dictionary is created containing:
     - `id`: A unique identifier (here, the index `i`).
     - `values`: The embedding vector values.
     - `metadata`: Metadata for each passage, including the original text (`'text'`).
   - These dictionaries are stored in the `vectors` list for later indexing.

3. **Indexing with Pinecone (or Similar Vector Database)**:
   - An index instance (`index`) is created to store vectors, associating each with a namespace (`"ns1"`).
   - Finally, the `index.upsert` function uploads the `vectors` to the index, making them queryable for tasks like similarity search or retrieval.

This setup allows efficient semantic searching and retrieval of the stored passages based on their embeddings.


In [ ]:
embeddings = pc.inference.embed(
    "multilingual-e5-large",
    inputs=df['Content'].tolist(),
    parameters={
        "input_type": "passage"
    }
)

vectors = []
for i, (d, e) in enumerate(zip(df['Content'], embeddings)):
    vectors.append({
        "id": str(i),
        "values": e['values'],
        "metadata": {'text': d}
    })

index = pc.Index('store')

index.upsert(
    vectors=vectors,
    namespace="ns1"
)

{'upserted_count': 2}

## Check which Gemini models are available for use

In [37]:
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

models/gemini-1.0-pro-latest
models/gemini-1.0-pro
models/gemini-pro
models/gemini-1.0-pro-001
models/gemini-1.0-pro-vision-latest
models/gemini-pro-vision
models/gemini-1.5-pro-latest
models/gemini-1.5-pro-001
models/gemini-1.5-pro-002
models/gemini-1.5-pro
models/gemini-1.5-pro-exp-0801
models/gemini-1.5-pro-exp-0827
models/gemini-1.5-flash-latest
models/gemini-1.5-flash-001
models/gemini-1.5-flash-001-tuning
models/gemini-1.5-flash
models/gemini-1.5-flash-exp-0827
models/gemini-1.5-flash-002
models/gemini-1.5-flash-8b
models/gemini-1.5-flash-8b-001
models/gemini-1.5-flash-8b-latest
models/gemini-1.5-flash-8b-exp-0827
models/gemini-1.5-flash-8b-exp-0924


## We'll be using Gemini 1.5-flash

In [38]:
model = genai.GenerativeModel('models/gemini-1.5-flash')

## Build the prompt for the LLM

This function builds a prompt for the LLM. It takes the original query,
    and the returned context, and asks the model to answer the question based only
    on what's in the context, not what's in its weights.

In [ ]:
def build_prompt(query: str, context: List[str]) -> str:
    """
    Builds a prompt for the LLM. #

    Args:
    query (str): The original query.
    context (List[str]): The context of the query, returned by embedding search.

    Returns:
    A prompt for the LLM (str).
    """

    base_prompt = {
        "content": "You are a human resources expert working for a renewed IT company. You are tasked with checking the CVs given to you and request thing to change and show the good things in the CVs. Answer only based on the context provided. Do not explain your answer.",
    }
    user_prompt = {
        "content": f" The question is '{query}'. Here is all the context you have:"
        f'{(" ").join(context)}',
    }

    # combine the prompts to output a single prompt string
    system = f"{base_prompt['content']} {user_prompt['content']}"

    return system


## Generating Gemini response

In [44]:
def get_gemini_response(query: str, context: List[str]) -> str:
    """
    Queries the Gemini API to get a response to the question.

    Args:
    query (str): The original query.
    context (List[str]): The context of the query, returned by embedding search.

    Returns:
    A response to the question.
    """

    response = model.generate_content(build_prompt(query, context))

    return response.text

## Chatting with the LLM

### Streamlit Application: Chat with Gemini

This code sets up an interactive chatbot application using Streamlit, integrating a large language model (LLM) to answer user queries based on relevant context retrieved from a vector index. The main steps include:

1. **Streamlit App Title**:
   - Sets the app title as "Chat with Gemini" using `st.title`.

2. **Session State Initialization**:
   - Initializes `st.session_state` variables for `query` and `response` to maintain query and response data across interactions, allowing the user’s input and the model’s response to persist.

3. **User Query Input**:
   - Provides a text input box for the user to enter a query.
   - If the "Submit" button is clicked, the code checks if the query is empty. If so, it prompts the user to enter a question.

4. **Embedding and Similarity Search**:
   - If a valid query is submitted, it uses the `pc.inference.embed` function to generate an embedding for the query, leveraging the `"multilingual-e5-large"` model for multilingual support.
   - The query embedding is used to search an index (`index.query`) for the top 3 most relevant passages (based on cosine similarity or other vector-based similarity metrics).
   - The `namespace="ns1"` parameter helps keep queries and contexts within a designated segment of the index, and `include_metadata=True` returns the text for each result.

5. **Response Generation**:
   - The relevant context passages from the index are passed, along with the user query, to `get_gemini_response`, which generates a response using the Gemini model.
   - The response is stored in `st.session_state.response` for display.

6. **Response Display**:
   - If there’s a response, it is displayed alongside the user’s original query, allowing for a conversational flow within the app.

This setup creates an intuitive interface where users can ask questions, and the application retrieves relevant context and generates informative responses using the Gemini model.


In [47]:
import streamlit as st


st.title("Chat with Gemini")

# Initialize session state for query and response
if "query" not in st.session_state:
    st.session_state.query = ""
if "response" not in st.session_state:
    st.session_state.response = ""

# Input box for user query
query = st.text_input("Query:", value=st.session_state.query)

if st.button("Submit"):
    if len(query) == 0:
        st.write("Please enter a question.")
    else:
        st.session_state.query = query
        st.write("Thinking...")

        x = pc.inference.embed(
            model="multilingual-e5-large",
            inputs=[query],
            parameters={
                "input_type": "query"
            }
        )

        results = index.query(
            namespace="ns1",
            vector=x[0].values,
            top_k=3,
            include_values=False,
            include_metadata=True
        )

        context = [match['metadata']['text'] for match in results['matches']]
        response = get_gemini_response(query, context)

        st.session_state.response = response

# Display the response
if st.session_state.response:
    st.write(f"Question: {st.session_state.query}")
    st.write(f"Response: {st.session_state.response}")

2024-11-11 20:22:12.884 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-11-11 20:22:12.918 
  command:

    streamlit run /Users/user/Library/Python/3.12/lib/python/site-packages/ipykernel_launcher.py [ARGUMENTS]
2024-11-11 20:22:12.919 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-11-11 20:22:12.920 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-11-11 20:22:12.920 Session state does not function when running a script without `streamlit run`
2024-11-11 20:22:12.921 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-11-11 20:22:12.922 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-11-11 20:22:12.922 Thread 'MainThread': missing Scri